# N-1 Vulnerability Analysis (IEEE 118-bus)

The full SC-DCOPF solves all operating states **together**: if any single state is
infeasible, the whole model is infeasible, and the IIS only reveals **one** conflict at a
time. To find out *exactly which* contingencies fail, we solve **each state in isolation** —
one small DCOPF per contingency. Each gives its own verdict (optimal / infeasible),
independent of the others.

This is a **diagnostic** study (not the model itself): it maps the network's weak points and
guides how to handle the infeasibilities (drop those contingencies, or add load shedding).

## 1. Load data (same processed inputs as the model)

In [1]:
import os
import pandas as pd
import gurobipy as gp
from gurobipy import GRB

ROOT   = ".."               # shared processed data lives one level up
SOURCE = "matpower"         # 'matpower' or 'scuc'
PROC   = f"{ROOT}/processed/{SOURCE}"
gens     = pd.read_csv(f"{PROC}/gens.csv")
loads    = pd.read_csv(f"{PROC}/loads.csv")
branches = pd.read_csv(f"{PROC}/branches.csv")
meta     = pd.read_csv(f"{PROC}/system_meta.csv").set_index("key")["value"]

BASE_MVA  = float(meta["base_mva"])
SLACK_BUS = int(meta["slack_bus"])

# sets
set_N = loads["bus"].tolist()
set_G = gens["gen_id"].tolist()
set_L = branches["branch_id"].tolist()

# parameters
par_pd    = dict(zip(loads["bus"], loads["pd_pu"]))
par_g_bus = dict(zip(gens["gen_id"], gens["bus"]))
par_pmax  = dict(zip(gens["gen_id"], gens["pmax_pu"]))
par_cost  = dict(zip(gens["gen_id"], gens["c1"]))
par_from  = dict(zip(branches["branch_id"], branches["from_bus"]))
par_to    = dict(zip(branches["branch_id"], branches["to_bus"]))
par_x     = dict(zip(branches["branch_id"], branches["x_pu"]))
par_rate  = dict(zip(branches["branch_id"], branches["rate_pu"]))
par_label = dict(zip(branches["branch_id"], branches["label"]))

# G_i, Omega+ , Omega-
set_gens_at = {i: [] for i in set_N}
for g in set_G:
    set_gens_at[par_g_bus[g]].append(g)
set_out = {i: [] for i in set_N}
set_in  = {i: [] for i in set_N}
for l in set_L:
    set_out[par_from[l]].append(l)
    set_in[par_to[l]].append(l)

# only non-radial lines are valid transmission contingencies
CT_lines = branches.loc[~branches["is_bridge"], "branch_id"].tolist()
print(f"Line contingencies to test (non-radial): {len(CT_lines)}")
print(f"Generator contingencies to test        : {len(set_G)}")

Line contingencies to test (non-radial): 177
Generator contingencies to test        : 19


## 2. Solve a single state in isolation

One helper function builds and solves **one** DCOPF. The element to remove is passed in:
- `out_gen=g`  disables generator `g` by setting its upper bound to 0 (it cannot produce).
- `out_line=l` disables line `l` by forcing its flow to 0 (instead of the DC flow law).

It returns just the solver status (2 = optimal, 3 = infeasible). Calling it with no argument
solves the base case.

In [2]:
def solve_single(out_line=None, out_gen=None):
    m = gp.Model()
    m.setParam("OutputFlag", 0)         # silent
    m.setParam("DualReductions", 0)     # report INFEASIBLE cleanly

    # variables (this state only, so no scenario index needed)
    # a generator that is 'out' gets upper bound 0 -> forced to 0
    p  = {g: m.addVar(lb=0.0, ub=(0.0 if g == out_gen else par_pmax[g])) for g in set_G}
    f  = {l: m.addVar(lb=-GRB.INFINITY, ub=GRB.INFINITY) for l in set_L}
    th = {i: m.addVar(lb=-GRB.INFINITY, ub=GRB.INFINITY) for i in set_N}

    m.setObjective(gp.quicksum(par_cost[g] * p[g] * BASE_MVA for g in set_G), GRB.MINIMIZE)

    # (3.2) nodal balance
    for i in set_N:
        gen_i = gp.quicksum(p[g] for g in set_gens_at[i])
        out_i = gp.quicksum(f[l] for l in set_out[i])
        in_i  = gp.quicksum(f[l] for l in set_in[i])
        m.addConstr(gen_i - par_pd[i] == out_i - in_i)

    # (3.4) thermal limits  +  (3.5/3.5b) DC flow law (line 'out' -> f = 0)
    for l in set_L:
        m.addConstr(f[l] >= -par_rate[l])
        m.addConstr(f[l] <=  par_rate[l])
        i, j = par_from[l], par_to[l]
        if l == out_line:
            m.addConstr(f[l] == 0.0)
        else:
            m.addConstr(f[l] == (1.0 / par_x[l]) * (th[i] - th[j]))

    # (3.6) reference bus
    m.addConstr(th[SLACK_BUS] == 0.0)

    m.optimize()
    return m.status

## 3. Sweep every contingency

Solve the base case, then every line contingency, then every generator contingency, and
record which ones are infeasible.

In [3]:
base_status = solve_single()
print("Base case:", "OK" if base_status == GRB.OPTIMAL else f"status {base_status}")

# scenario_id must match the model's numbering (sc-dcopf.ipynb): base = 0, then the
# lines of CT_lines (ids 1..), then the generators. It is a pure-Python label, fully
# deterministic and independent of Gurobi. We rebuild it here so this table can be
# cross-referenced with the model and its IIS by scenario_id.
records = []
sid = 1  # id 0 is the base case (not a contingency), so contingencies start at 1

# line contingencies (same order as in the model)
for l in CT_lines:
    status = solve_single(out_line=l)
    records.append({"scenario_id": sid, "type": "line", "element_id": l,
                    "element": par_label[l], "feasible": status == GRB.OPTIMAL})
    sid += 1

# generator contingencies (after all lines, same as in the model)
for g in set_G:
    status = solve_single(out_gen=g)
    records.append({"scenario_id": sid, "type": "gen", "element_id": g,
                    "element": f"bus {par_g_bus[g]} ({par_pmax[g]*BASE_MVA:.0f} MW)",
                    "feasible": status == GRB.OPTIMAL})
    sid += 1

vuln = pd.DataFrame(records)
n_inf = (~vuln["feasible"]).sum()
print(f"\nTested {len(vuln)} contingencies | {n_inf} INFEASIBLE")

Set parameter WLSAccessID
Set parameter WLSSecret
Set parameter LicenseID to value 2855642
Academic license 2855642 - for non-commercial use only - registered to ra___@poli.ufrj.br
Base case: OK

Tested 196 contingencies | 3 INFEASIBLE


## 4. The weak points

In [4]:
weak = vuln[~vuln["feasible"]].reset_index(drop=True)
print("Infeasible contingencies:")
weak

Infeasible contingencies:


,type,element_id,element,feasible
0,line,7,8-5,False
1,line,50,38-37,False
2,gen,0,bus 10 (505 MW),False


## 5. Save the vulnerability map

Full table (all contingencies + feasibility) and the weak-points-only table, for the record.

In [5]:
OUT = f"results/{SOURCE}/vulnerability"
os.makedirs(OUT, exist_ok=True)
vuln.to_csv(f"{OUT}/vulnerability_full.csv", index=False)
weak.to_csv(f"{OUT}/vulnerability_weak_points.csv", index=False)
print(f"Saved to {OUT}/ ({len(vuln)} tested, {len(weak)} infeasible).")

Saved to results/vulnerability/ (196 tested, 3 infeasible).
